In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_pickle(r"C:\Users\Almog\Desktop\Data Science\Projects\2026\Customer Personality Analysis\Pickle files\FE_MC1.pkl")
df.head()

,Year_Birth,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,...,tenure_years,Education_ord,Marital_Status_encoded,total_spending,total_purchases,is_web_dominant,value_optimizer,spending_to_income_ratio,family_load,is_long_tenure
0,1957,58138.0,0,0,58,635,88,546,172,88,...,1.815,2,2,1617,12,1,0,0.027813,0,1
1,1954,46344.0,1,1,38,11,1,6,2,1,...,0.309,2,2,27,3,0,0,0.000583,1,0
2,1965,71613.0,0,0,26,426,49,127,111,21,...,0.854,2,1,776,18,0,0,0.010836,0,0
3,1984,26646.0,1,0,26,11,4,20,10,3,...,0.381,2,1,53,6,0,0,0.001989,1,0
4,1981,58293.0,1,0,94,173,43,118,46,27,...,0.441,4,0,422,11,0,0,0.007239,1,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 33 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Year_Birth                2240 non-null   int64  
 1   Income                    2240 non-null   float64
 2   Kidhome                   2240 non-null   int64  
 3   Teenhome                  2240 non-null   int64  
 4   Recency                   2240 non-null   int64  
 5   MntWines                  2240 non-null   int64  
 6   MntFruits                 2240 non-null   int64  
 7   MntMeatProducts           2240 non-null   int64  
 8   MntFishProducts           2240 non-null   int64  
 9   MntSweetProducts          2240 non-null   int64  
 10  MntGoldProds              2240 non-null   int64  
 11  NumDealsPurchases         2240 non-null   int64  
 12  NumWebPurchases           2240 non-null   int64  
 13  NumCatalogPurchases       2240 non-null   int64  
 14  NumStore

#### Featrue Selection

Because clustering has no target variable, we cannot use classifiers to evaluate predictive importance. Instead, we measure how much each feature contributes to cluster structure. The code above first calculates a baseline silhouette score using all features. Then it removes each feature one at a time, reruns clustering, and measures the new silhouette score. If removing a feature reduces the silhouette score, the feature is important for forming clusters; if removing it improves or barely changes the score, the feature may be redundant or noisy and can potentially be dropped. This gives you a practical unsupervised feature selection ranking similar in spirit to your supervised voting approach.

**Silhouette_without_feature**
The silhouette score obtained after removing that feature and rerunning clustering. It measures how well the clusters are separated when that feature is excluded.

**Impact**
The difference between the baseline silhouette score (with all features) and the silhouette score after removing the feature.

- Positive Impact → removing the feature worsens clustering → the feature is important.

- Negative Impact → removing the feature improves clustering → the feature may be noise or redundant.

In [6]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Assume df is already numeric and ready
X = df.copy()

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Baseline clustering
kmeans = KMeans(n_clusters=4, random_state=42)
labels = kmeans.fit_predict(X_scaled)
baseline_score = silhouette_score(X_scaled, labels)

results = []

# Leave-one-feature-out evaluation
for col in X.columns:
    X_reduced = X.drop(columns=[col])
    
    scaler = StandardScaler()
    X_scaled_reduced = scaler.fit_transform(X_reduced)

    kmeans = KMeans(n_clusters=4, random_state=42)
    labels = kmeans.fit_predict(X_scaled_reduced)

    score = silhouette_score(X_scaled_reduced, labels)

    results.append({
        "Feature": col,
        "Silhouette_without_feature": score,
        "Impact": baseline_score - score
    })

selection_df = pd.DataFrame(results).sort_values("Impact", ascending=False)



In [7]:


# Copy results
feature_summary = selection_df.copy()

# Function to categorize importance
def categorize_feature(impact):
    if impact >= 0.08:
        return "Strongly Influential"
    elif impact >= 0.01:
        return "Moderately Contributing"
    elif impact >= 0:
        return "Low Influence"
    else:
        return "Potential Noise"

# Apply categorization
feature_summary["Category"] = feature_summary["Impact"].apply(categorize_feature)

# Add rank column
feature_summary["Rank"] = feature_summary["Impact"].rank(ascending=False).astype(int)

# Sort by rank
feature_summary = feature_summary.sort_values("Rank")

feature_summary

,Feature,Silhouette_without_feature,Impact,Category,Rank
23,tenure_years,0.081856,0.104735,Strongly Influential,1
1,Income,0.092270,0.094320,Strongly Influential,2
11,NumDealsPurchases,0.093556,0.093034,Strongly Influential,3
30,spending_to_income_ratio,0.094172,0.092418,Strongly Influential,4
21,Complain,0.094765,0.091825,Strongly Influential,5
10,MntGoldProds,0.096179,0.090412,Strongly Influential,6
16,AcceptedCmp3,0.098035,0.088555,Strongly Influential,7
20,AcceptedCmp2,0.098198,0.088392,Strongly Influential,8
0,Year_Birth,0.098789,0.087801,Strongly Influential,9
4,Recency,0.100711,0.085879,Strongly Influential,10


Since clustering is an **unsupervised learning task**, there is no target variable to evaluate feature importance directly.  
To assess how each feature contributes to the clustering structure, we performed a **Leave-One-Feature-Out (LOFO) analysis** using K-Means clustering and the **Silhouette Score** as the evaluation metric.

The process was conducted as follows:

1. Compute a **baseline silhouette score** using all available features.
2. Iteratively **remove one feature at a time** from the dataset.
3. Re-run the clustering model on the reduced feature set.
4. Measure the **new silhouette score** obtained without that feature.
5. Compute the **Impact** as:


**Interpretation:**

- **Positive Impact** → Removing the feature **reduces cluster quality**, meaning the feature contributes positively to cluster separation.
- **Near-Zero Impact** → The feature has **little influence** on clustering structure.
- **Negative Impact** → Removing the feature **improves clustering**, suggesting the feature may introduce noise or redundancy.

This approach provides an **unsupervised proxy for feature importance**, helping identify which variables meaningfully shape the geometry of the clustering space.

---

**Key Observations**

**Strongly Influential Features**

Several features show a **large positive impact**, indicating that they are important for distinguishing clusters:

These variables appear to capture **important behavioral and economic dimensions** such as customer lifecycle stage, financial capacity, promotional sensitivity, and engagement with the brand.

---

**Moderately Contributing Features**

Some features contribute to cluster formation but with **smaller influence**:

These features still add information but may partially overlap with other variables.

---

**Low Influence Features**

A number of variables show **very small impact**, meaning removing them barely changes the clustering quality:

These features likely carry information that is already captured by other variables.

---

**Potential Noise Features**

Some features produce **negative impact**, meaning the clustering slightly improves when they are removed:

These variables may introduce **noise, redundancy, or weak signals** relative to the other features.

---

**Conclusion & Bottom Line**

The analysis indicates that **economic capacity, customer tenure, promotional engagement, and purchasing behavior** are the most important dimensions shaping the clustering structure.

Based on the LOFO results, the following features appear to **degrade or minimally contribute to clustering quality** and are candidates for removal:

**Features to Drop**

- `AcceptedCmp1`
- `Education_ord`
- `Response`
- `NumWebVisitsMonth`
- `MntSweetProducts`

These variables either introduce noise or provide limited additional information beyond what is already captured by stronger behavioral and financial features.

Removing them helps **simplify the feature space**, reduce redundancy, and improve the clarity of the clustering geometry.

The remaining features will be retained for the next stage of the workflow, which involves **feature transformation and scaling prior to running the final clustering models**.

In [18]:
features_to_drop = [
    'AcceptedCmp1',
    'Education_ord',
    'Response',
    'NumWebVisitsMonth',
    'MntSweetProducts'
]

df = df.drop(columns=features_to_drop)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 28 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Year_Birth                2240 non-null   int64  
 1   Income                    2240 non-null   float64
 2   Kidhome                   2240 non-null   int64  
 3   Teenhome                  2240 non-null   int64  
 4   Recency                   2240 non-null   int64  
 5   MntWines                  2240 non-null   int64  
 6   MntFruits                 2240 non-null   int64  
 7   MntMeatProducts           2240 non-null   int64  
 8   MntFishProducts           2240 non-null   int64  
 9   MntGoldProds              2240 non-null   int64  
 10  NumDealsPurchases         2240 non-null   int64  
 11  NumWebPurchases           2240 non-null   int64  
 12  NumCatalogPurchases       2240 non-null   int64  
 13  NumStorePurchases         2240 non-null   int64  
 14  Accepted

In [20]:
df.to_pickle(r"C:\Users\Almog\Desktop\Data Science\Projects\2026\Customer Personality Analysis\Pickle files\FS_MC1.pkl")